# Komoot Surface Scraper — par annee
Change juste ANNEE pour scraper une annee a la fois.

In [19]:
import os, time, json, re
import pandas as pd
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

GPX_DIR     = '/Users/arthurdeletang/Desktop/Stage M1/Code/data/gpx_files_2'
OUTPUT_BASE = '/Users/arthurdeletang/Desktop/Stage M1/Code/data/matching'

In [20]:
ANNEE = 2025  # <-- change cette ligne pour chaque annee

CHECKPOINT = os.path.join(OUTPUT_BASE, 'all', f'komoot_checkpoint_all.json')
OUTPUT_CSV  = os.path.join(OUTPUT_BASE, 'all', f'komoot_surface_all.csv')

path = os.path.join(OUTPUT_BASE, 'all', f'matching_{ANNEE}.csv')
df_matching = pd.read_csv(path)
df_matching = df_matching[df_matching['statut'] == 'match']
gpx_all = df_matching['fichier_gpx'].dropna().unique().tolist()
print(f'{ANNEE} : {len(gpx_all)} courses')

results = []
already_done = set()
if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT) as f:
        all_results = json.load(f)
    results = [r for r in all_results if r.get('route_url') and len(r) > 2]
    already_done = {r['fichier_gpx'] for r in results}
    echecs = [r['fichier_gpx'] for r in all_results if not r.get('route_url') or len(r) <= 2]
    print(f'Checkpoint : {len(already_done)} succes, {len(echecs)} echecs a re-essayer')

todo = [f for f in sorted(gpx_all) if f not in already_done]
print(f'Restant : {len(todo)}')

2025 : 906 courses
Checkpoint : 1837 succes, 6 echecs a re-essayer
Restant : 6


In [21]:
options = Options()
options.add_argument('--window-size=1440,900')
options.add_argument('--use-gl=swiftshader')
options.add_argument('--enable-unsafe-swiftshader')
options.add_argument('--enable-webgl')
options.add_argument('--ignore-gpu-blocklist')

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

driver.get('https://www.komoot.com/login')
print('Connecte-toi a Komoot, puis lance la cellule suivante.')

Connecte-toi a Komoot, puis lance la cellule suivante.


In [22]:
def find_gpx_path(fname, gpx_dir):
    full_path = os.path.join(gpx_dir, fname)
    if os.path.exists(full_path):
        return full_path
    # Essayer avec ' - ' au lieu de ' / '
    replaced = fname.replace(' / ', ' - ')
    repl_path = os.path.join(gpx_dir, replaced)
    if os.path.exists(repl_path):
        return repl_path
    # Essayer le nom tronque avant le '/'
    truncated = fname.split(' / ')[0].strip() + '.gpx'
    trunc_path = os.path.join(gpx_dir, truncated)
    if os.path.exists(trunc_path):
        return trunc_path
    return None

def wait_for_text(text, timeout=30):
    try:
        WebDriverWait(driver, timeout).until(
            lambda d: text in d.find_element(By.TAG_NAME, 'body').text
        )
        return True
    except Exception:
        return False

def parse_km(text):
    text = text.replace('\u00a0', ' ').strip()
    if '< 100' in text:
        return 0.0
    parts = text.split(' ')
    try:
        val = float(parts[0].replace(',', '.'))
        unit = parts[1] if len(parts) > 1 else 'km'
        return round(val / 1000 if unit == 'm' else val, 3)
    except Exception:
        return None

def upload_and_save(filepath):
    driver.get('https://www.komoot.com/upload')
    time.sleep(2)

    try:
        file_input = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, "//input[@type='file']"))
        )
        driver.execute_script("arguments[0].style.display = 'block';", file_input)
        file_input.send_keys(filepath)
    except Exception as e:
        print(f'  Upload introuvable : {e}')
        return None

    # Attendre Import to Plan a Route OU erreur
    try:
        WebDriverWait(driver, 60).until(
            lambda d: 'Import to Plan a Route' in d.find_element(By.TAG_NAME, 'body').text
                      or "couldn't be imported" in d.find_element(By.TAG_NAME, 'body').text
                      or 'sorry' in d.find_element(By.TAG_NAME, 'body').text.lower()
        )
    except Exception:
        print('  Timeout attente dialogue')
        return None

    body = driver.find_element(By.TAG_NAME, 'body').text
    if "couldn't be imported" in body or 'sorry' in body.lower():
        print('  Fichier rejete par Komoot')
        return None

    if 'Import to Plan a Route' not in body:
        print('  Import dialog non detecte')
        return None

    # Etape 1 : Next
    driver.execute_script("document.querySelectorAll('a[aria-label=\"Next\"]')[0]?.click();")
    print('  Etape 1 clic Next')

    if not wait_for_text('Road cycling', timeout=30):
        print('  Page Sport non detectee')
        return None
    print('  Etape 1 OK -> Page Sport')

    # Etape 2 : Next
    driver.execute_script("document.querySelectorAll('a[aria-label=\"Next\"]')[0]?.click();")
    print('  Etape 2 clic Next')

    # Attendre Resolve Routing OU Route matches OU Route deviates OU Route Saved
    try:
        WebDriverWait(driver, 30).until(
            lambda d: 'Resolve Routing' in d.find_element(By.TAG_NAME, 'body').text
                      or 'Route matches to known ways' in d.find_element(By.TAG_NAME, 'body').text
                      or 'Route deviates from known ways' in d.find_element(By.TAG_NAME, 'body').text
                      or 'Route Saved' in d.find_element(By.TAG_NAME, 'body').text
        )
    except Exception:
        print('  Aucune page intermediaire detectee')
        print('  Body:', driver.find_element(By.TAG_NAME, 'body').text[:300])
        return None

    body = driver.find_element(By.TAG_NAME, 'body').text

    if 'Resolve Routing' in body or 'Route matches to known ways' in body or 'Route deviates from known ways' in body:
        if 'Resolve Routing' in body:
            print('  Resolve Routing detecte')
        elif 'Route deviates from known ways' in body:
            print('  Route deviates detecte')
        else:
            print('  Route matches detecte')
        driver.execute_script("document.querySelectorAll('a[aria-label=\"Save\"]')[0]?.click();")
        print('  Etape 3 clic Save')
        if not wait_for_text('Route Saved', timeout=30):
            print('  Route Saved non detecte apres Save')
            return None

    print('  Route Saved OK')
    driver.execute_script("document.querySelectorAll('a.css-zi6wow')[0]?.click();")

    try:
        WebDriverWait(driver, 15).until(lambda d: '/tour/' in d.current_url)
    except Exception:
        print('  URL /tour/ non detectee')
        return None

    time.sleep(1)
    return driver.current_url


def scrape_surface(route_url):
    driver.get(route_url)
    try:
        WebDriverWait(driver, 15).until(
            lambda d: len(d.find_elements(By.CSS_SELECTOR, 'div.css-esdg3g')) > 0
        )
    except Exception:
        print('  Section Way Types non chargee')
        return {}
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight / 2);')
    time.sleep(1)
    entry = {}
    try:
        rows = driver.find_elements(By.CSS_SELECTOR, 'div.css-esdg3g')
        for row in rows:
            try:
                label_el = row.find_element(By.CSS_SELECTOR, 'button.css-1xu3avq')
                value_el = row.find_element(By.CSS_SELECTOR, 'p.css-1wxtat5')
                label = label_el.text.strip().rstrip(':').lower().replace(' ', '_')
                if not label:
                    continue
                entry[f'{label}_km'] = parse_km(value_el.text)
            except Exception:
                continue
    except Exception as e:
        print(f'  scrape_surface : {e}')
    return entry


def save_checkpoint():
    tmp = CHECKPOINT + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    os.replace(tmp, CHECKPOINT)

print('Fonctions chargees.')

Fonctions chargees.


In [23]:
for idx, fname in enumerate(tqdm(todo)):
    filepath = find_gpx_path(fname, GPX_DIR)
    entry = {'fichier_gpx': fname}

    if not filepath:
        print(f'  Fichier introuvable : {fname}')
        results.append(entry)
        save_checkpoint()
        continue

    route_url = None
    try:
        route_url = upload_and_save(filepath)
    except Exception as e:
        print(f'  Exception : {e}')

    if route_url:
        entry['route_url'] = route_url
        entry.update(scrape_surface(route_url))
        print(f'  OK : {fname}')
    else:
        print(f'  ECHEC FINAL : {fname}')

    results.append(entry)
    save_checkpoint()

print('Termine.')
pd.DataFrame(results)

  0%|          | 0/6 [00:00<?, ?it/s]

  Fichier introuvable : 2025 GP Vorarlberg p/b Radhaus Rankweil.gpx
  Etape 1 clic Next
  Etape 1 OK -> Page Sport
  Etape 2 clic Next
  Route matches detecte
  Etape 3 clic Save
  Route Saved OK


 33%|███▎      | 2/6 [00:48<01:37, 24.39s/it]

  OK : 2025 Lidl Deutschland Tour Prologue.gpx
  Etape 1 clic Next
  Etape 1 OK -> Page Sport
  Etape 2 clic Next
  Resolve Routing detecte
  Etape 3 clic Save
  Route Saved OK


 50%|█████     | 3/6 [01:17<01:18, 26.04s/it]

...

 83%|████████▎ | 5/6 [02:19<00:28, 28.34s/it]

  OK : 2025 Tour du Limousin-Perigord - Nouvelle Aquitaine Stage 1.gpx
  Etape 1 clic Next
  Etape 1 OK -> Page Sport
  Etape 2 clic Next
  Resolve Routing detecte
  Etape 3 clic Save
  Route Saved OK


100%|██████████| 6/6 [02:40<00:00, 26.81s/it]

  OK : 2025 Tour du Limousin-Perigord - Nouvelle Aquitaine Stage 2.gpx
Termine.


,fichier_gpx,route_url,road_km,cycleway_km,street_km,unknown_km,state_road_km,off-grid_(unknown)_km,path_km,singletrack_km,asphalt_km,paved_km,cobblestones_km,unpaved_km,compacted_gravel_km,access_road_km,alpine_km
0,2025 ADAC Cyclassics.gpx,https://www.komoot.com/tour/2950103693,104.0,59.30,23.000,3.010,5.810,2.850,0.611,0.356,171.00,30.80,2.68,0.403,NaN,NaN,NaN
1,2025 Amstel Gold Race.gpx,https://www.komoot.com/tour/2950105131,121.0,82.50,7.270,5.830,0.852,5.380,0.278,0.198,225.00,18.80,6.78,0.210,NaN,NaN,NaN
2,2025 Bretagne Classic - CIC.gpx,https://www.komoot.com/tour/2950106378,238.0,1.43,7.280,0.569,14.500,0.569,NaN,NaN,202.00,59.10,NaN,NaN,NaN,NaN,NaN
3,2025 Cadel Evans Great Ocean Road Race.gpx,https://www.komoot.com/tour/2950107577,103.0,29.30,3.920,0.504,4.410,0.504,0.000,NaN,94.30,46.80,NaN,NaN,NaN,NaN,NaN
4,2025 Classic Brugge-De Panne.gpx,https://www.komoot.com/tour/2950108889,70.1,66.70,31.800,0.757,9.750,NaN,0.357,0.731,142.00,47.60,5.24,0.731,0.321,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1838,2025 Lidl Deutschland Tour Prologue.gpx,https://www.komoot.com/tour/3006735169,NaN,1.36,1.630,NaN,NaN,NaN,NaN,NaN,2.99,NaN,NaN,NaN,NaN,NaN,NaN
1839,2025 Renewi Tour Stage 1.gpx,https://www.komoot.com/tour/3006736499,58.1,45.60,8.120,16.400,0.000,13.000,1.870,0.776,136.00,24.90,6.29,0.795,0.436,0.163,NaN
1840,2025 Tour Bitwa Warszawska Stage 1.gpx,https://www.komoot.com/tour/3006737333,143.0,NaN,28.300,NaN,NaN,NaN,NaN,NaN,168.00,3.33,NaN,NaN,NaN,NaN,NaN
1841,2025 Tour du Limousin-Perigord - Nouvelle Aqui...,https://www.komoot.com/tour/3006738700,180.0,NaN,0.564,NaN,3.900,NaN,NaN,NaN,174.00,9.88,NaN,NaN,NaN,NaN,NaN


In [24]:
driver.quit()
df_komoot = pd.DataFrame(results).fillna(0)
df_komoot.to_csv(OUTPUT_CSV, index=False)
print(f'Sauvegarde : {OUTPUT_CSV}')
print(f'Shape : {df_komoot.shape}')
df_komoot

Sauvegarde : /Users/arthurdeletang/Desktop/Stage M1/Code/data/matching/all/komoot_surface_all.csv
Shape : (1843, 17)


,fichier_gpx,route_url,road_km,cycleway_km,street_km,unknown_km,state_road_km,off-grid_(unknown)_km,path_km,singletrack_km,asphalt_km,paved_km,cobblestones_km,unpaved_km,compacted_gravel_km,access_road_km,alpine_km
0,2025 ADAC Cyclassics.gpx,https://www.komoot.com/tour/2950103693,104.0,59.30,23.000,3.010,5.810,2.850,0.611,0.356,171.00,30.80,2.68,0.403,0.000,0.000,0.0
1,2025 Amstel Gold Race.gpx,https://www.komoot.com/tour/2950105131,121.0,82.50,7.270,5.830,0.852,5.380,0.278,0.198,225.00,18.80,6.78,0.210,0.000,0.000,0.0
2,2025 Bretagne Classic - CIC.gpx,https://www.komoot.com/tour/2950106378,238.0,1.43,7.280,0.569,14.500,0.569,0.000,0.000,202.00,59.10,0.00,0.000,0.000,0.000,0.0
3,2025 Cadel Evans Great Ocean Road Race.gpx,https://www.komoot.com/tour/2950107577,103.0,29.30,3.920,0.504,4.410,0.504,0.000,0.000,94.30,46.80,0.00,0.000,0.000,0.000,0.0
4,2025 Classic Brugge-De Panne.gpx,https://www.komoot.com/tour/2950108889,70.1,66.70,31.800,0.757,9.750,0.000,0.357,0.731,142.00,47.60,5.24,0.731,0.321,0.000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1838,2025 Lidl Deutschland Tour Prologue.gpx,https://www.komoot.com/tour/3006735169,0.0,1.36,1.630,0.000,0.000,0.000,0.000,0.000,2.99,0.00,0.00,0.000,0.000,0.000,0.0
1839,2025 Renewi Tour Stage 1.gpx,https://www.komoot.com/tour/3006736499,58.1,45.60,8.120,16.400,0.000,13.000,1.870,0.776,136.00,24.90,6.29,0.795,0.436,0.163,0.0
1840,2025 Tour Bitwa Warszawska Stage 1.gpx,https://www.komoot.com/tour/3006737333,143.0,0.00,28.300,0.000,0.000,0.000,0.000,0.000,168.00,3.33,0.00,0.000,0.000,0.000,0.0
1841,2025 Tour du Limousin-Perigord - Nouvelle Aqui...,https://www.komoot.com/tour/3006738700,180.0,0.00,0.564,0.000,3.900,0.000,0.000,0.000,174.00,9.88,0.00,0.000,0.000,0.000,0.0


In [25]:
import pandas as pd
import os

OUTPUT_BASE = '/Users/arthurdeletang/Desktop/Stage M1/Code/data/matching'

manquants = []
for year in range(2017, 2026):
    path = os.path.join(OUTPUT_BASE, 'all', f'matching_{year}.csv')
    if os.path.exists(path):
        df = pd.read_csv(path)
        df = df[df['statut'] == 'match']
        m = df[df['fichier_gpx'].str.contains(' / ', na=False)]['fichier_gpx'].unique()
        if len(m) > 0:
            print(f'{year} : {len(m)} fichiers manquants')
            manquants.extend(m)

print(f'\nTotal : {len(manquants)} fichiers avec / dans le nom')

2017 : 10 fichiers manquants
2018 : 17 fichiers manquants
2019 : 16 fichiers manquants
2020 : 1 fichiers manquants
2021 : 15 fichiers manquants
2022 : 19 fichiers manquants
2023 : 14 fichiers manquants
2024 : 17 fichiers manquants
2025 : 18 fichiers manquants

Total : 127 fichiers avec / dans le nom


In [26]:
import os, json
import pandas as pd

OUTPUT_BASE = '/Users/arthurdeletang/Desktop/Stage M1/Code/data/matching'

# Charger tous les checkpoints
all_results = []

# 2017-2023 : un fichier par année
for year in range(2017, 2024):
    path = os.path.join(OUTPUT_BASE, 'all', f'komoot_checkpoint_{year}.json')
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        succes = [r for r in data if r.get('route_url') and len(r) > 2]
        all_results.extend(succes)
        print(f'{year} : {len(succes)} succes')

# 2024-2025 : fichier commun
path_all = os.path.join(OUTPUT_BASE, 'all', 'komoot_checkpoint_all.json')
if os.path.exists(path_all):
    with open(path_all) as f:
        data = json.load(f)
    succes = [r for r in data if r.get('route_url') and len(r) > 2]
    all_results.extend(succes)
    print(f'2024-2025 : {len(succes)} succes')

# Dedupliquer sur fichier_gpx
df_komoot = pd.DataFrame(all_results)
df_komoot = df_komoot.drop_duplicates(subset='fichier_gpx', keep='last')
print(f'\nTotal unique : {len(df_komoot)} courses')
print(f'Colonnes : {list(df_komoot.columns)}')

2017 : 583 succes
2018 : 1016 succes
2019 : 1044 succes
2020 : 444 succes
2021 : 693 succes
2022 : 804 succes
2023 : 917 succes
2024-2025 : 1842 succes

Total unique : 7343 courses
Colonnes : ['fichier_gpx', 'route_url', 'road_km', 'street_km', 'state_road_km', 'cycleway_km', 'off-grid_(unknown)_km', 'access_road_km', 'asphalt_km', 'paved_km', 'cobblestones_km', 'unknown_km', 'path_km', 'unpaved_km', 'singletrack_km', 'compacted_gravel_km', '_km', 'alpine_km', 'ferry_km', 'alpine_path_km']


In [27]:
# Sauvegarder le CSV Komoot fusionné
output_path = os.path.join(OUTPUT_BASE, 'all', 'komoot_surface_all.csv')
df_komoot.to_csv(output_path, index=False)
print(f'Sauvegarde : {output_path}')
print(f'Shape : {df_komoot.shape}')

Sauvegarde : /Users/arthurdeletang/Desktop/Stage M1/Code/data/matching/all/komoot_surface_all.csv
Shape : (7343, 20)


In [28]:
import os
gpx_dir = "/Users/arthurdeletang/Desktop/Stage M1/Code/data/gpx_files_2"
files_2025 = [f for f in os.listdir(gpx_dir) if f.startswith("2025")]
print(sorted(files_2025))

['2025 4 Jours de Dunkerque - Grand Prix des Hauts de France Stage 1.gpx', '2025 4 Jours de Dunkerque - Grand Prix des Hauts de France Stage 2.gpx', '2025 4 Jours de Dunkerque - Grand Prix des Hauts de France Stage 3.gpx', '2025 4 Jours de Dunkerque - Grand Prix des Hauts de France Stage 4.gpx', '2025 4 Jours de Dunkerque - Grand Prix des Hauts de France Stage 5.gpx', '2025 4 Jours de Dunkerque .gpx', '2025 A Travers les Hauts de France.gpx', '2025 ADAC Cyclassics.gpx', '2025 Ain Bugey Valromey Tour Stage 1.gpx', '2025 Ain Bugey Valromey Tour Stage 2.gpx', '2025 Ain Bugey Valromey Tour Stage 3.gpx', '2025 Ain Bugey Valromey Tour Stage 4.gpx', '2025 Ain Bugey Valromey Tour Stage 5.gpx', '2025 AlUla Tour Stage 1.gpx', '2025 AlUla Tour Stage 2.gpx', '2025 AlUla Tour Stage 3.gpx', '2025 AlUla Tour Stage 4.gpx', '2025 AlUla Tour Stage 5.gpx', '2025 Alpes Gresivaudan Classic.gpx', '2025 Alpes Isere Tour Stage 1.gpx', '2025 Alpes Isere Tour Stage 2.gpx', '2025 Alpes Isere Tour Stage 3.gpx', '

In [29]:
import os

gpx_dir = "/Users/arthurdeletang/Desktop/Stage M1/Code/data/gpx_files_2"
corrompus = []

for fname in os.listdir(gpx_dir):
    if not fname.endswith('.gpx'):
        continue
    path = os.path.join(gpx_dir, fname)
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        debut = f.read(200)
    if '<!DOCTYPE' in debut or '<html' in debut or 'Bad gateway' in debut:
        corrompus.append(fname)

print(f"{len(corrompus)} fichiers corrompus")
for f in sorted(corrompus):
    print(f)

124 fichiers corrompus
2017 Giro d'Italia Rest day.gpx
2017 Koga Slag om Norg Rest day.gpx
2017 Tour de France Rest day.gpx
2017 Tour de l'Avenir Rest day.gpx
2017 Volta a Portugal em Bicicleta Santander Totta Rest day.gpx
2017 Vuelta a Espana Rest day.gpx
2018 Africa Cup - ITT.gpx
2018 Africa Cup - TTT.gpx
2018 Africa Cup.gpx
2018 Burmese Road National Championships - ITT.gpx
2018 Burmese Road National Championships.gpx
2018 Giro d'Italia Rest day.gpx
2018 Qatar Road National Championships.gpx
2018 Tour de France Rest day.gpx
2018 Tour de l'Espoir Rest day.gpx
2018 Tour of China II Rest day.gpx
2018 Tour of Fuzhou Stage 4.gpx
2018 Volta a Portugal em Bicicleta Santander Totta Rest day.gpx
2018 Vuelta Ciclista a la Provincia de San Juan Rest day.gpx
2018 Vuelta a Espana Rest day.gpx
2019 Belizean Road National Championships.gpx
2019 COPACI Campeonato Panamericano de Ruta Junior - ITT.gpx
2019 Chinese Road National Championships - TTT.gpx
2019 Chinese Road National Championships.gpx
201